# Stage 1 — FL Fusion Core: Projection Heads + FedProto (+ FedTGP upgrade)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

PROJECT = "/content/drive/MyDrive/Depression_FL_Project"

# Stage 0 output
LABELS_CSV = f"{PROJECT}/fl/labels_v1/fl_harmonized_labels.csv"

EEG_EMBED_DIR = f"{PROJECT}/data/embeddings"   # eeg_embeddings_{train,val,test}.npy + eeg_subj_ids_* + eeg_labels_*

TEXT_EMBED_NPZ = f"{PROJECT}/output/PDCH/processed_v1/embeddings/pdch_text_emb_full_dialogue.npz"

AUDIO_TURN_DIR = f"{PROJECT}/audio/v1/turn_embeddings"
AUDIO_LOCK_LAYER = 10   # matches the locked model's chosen WavLM layer

OUT = f"{PROJECT}/fl/fusion_v1"
import os, glob, json
import numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)

for p in (LABELS_CSV, EEG_EMBED_DIR, TEXT_EMBED_NPZ, AUDIO_TURN_DIR):
    print(f"  exists={os.path.exists(p)}  {p}")

In [ ]:
!pip -q install torch --upgrade 2>/dev/null
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import math
import matplotlib.pyplot as plt

In [ ]:

# Classification task: binary depressed/not  vs  3-class severity
TASK = "severity3"           # "binary" | "severity3"
# to all silos, or this dict for a per-silo cut.
# vs not' -- state this explicitly in the methods section.
BINARY_POSITIVE_FROM = {"eeg": 1, "audio": 1, "text": 2}
N_CLASSES    = 2 if TASK == "binary" else 3

# land near 0.5 (no change); the imbalanced text silo benefits most.
USE_THRESHOLD_TUNING = True
THRESHOLD_GRID       = 41    # candidate thresholds spanning (0.05 .. 0.95)
SHARED_DIM   = 512

# More local optimization per round
LOCAL_EPOCHS = 10

N_ROUNDS     = 100
EARLY_STOP_PATIENCE = 20
WARMUP_ROUNDS = 15

LR_HEAD      = 1e-4

N_CLIENTS    = 3

HIDDEN_DIM   = 256
N_HEAD_LAYERS = 2
HEAD_DROPOUT = 0.2

# Performance levers added for this run
USE_INPUT_STANDARDIZATION = True   # z-score head inputs per silo (TRAIN stats only)
BATCH_SIZE                = 32      # mini-batch local training (was full-batch)
AUGMENT_NOISE_STD         = 0.05    # Gaussian embedding jitter -> cheap augmentation
USE_CLASS_WEIGHTING       = True    # inverse-freq weighting in the prototype loss (macro-F1)
PROTO_TEMPERATURE         = 0.1     # prototype contrastive temperature

GRAD_CLIP    = 1.0

# Aggregation
USE_SAMPLE_SIZE_WEIGHTING = False
USE_CLIENT_RELIABILITY_WEIGHTING = True

# Losses
USE_COMPACTNESS = True
COMPACT_LAMBDA  = 0.20

USE_CLASSIFIER_HEAD = True       # add a trained classifier on the projected rep
PREDICT_WITH        = 'classifier'   # 'classifier' | 'prototype'  (eval + early-stop metric)
CE_WEIGHT           = 1.0        # weight of the classification loss in the local objective
USE_FOCAL_LOSS      = True       # focal vs plain (class-weighted) cross-entropy
FOCAL_GAMMA         = 1.5        # focal focusing parameter (0 -> plain weighted CE)
CE_LABEL_SMOOTHING  = 0.05       # label smoothing for plain CE (ignored under focal)
USE_MIXUP           = True       # mixup augmentation on the classification path only
MIXUP_ALPHA         = 0.2        # Beta(alpha,alpha) mixing strength

N_SEEDS = 5

# Prototype Stability
USE_ETF_INIT = True

PROTO_EMA_MOMENTUM = 0.95
GATE_ADAPT_MOMENTUM = 0.95

GATE_MIN = 0.05
GATE_MAX = 0.95

# Modality Personalization
# Audio should stay mostly local
# Text moderately local
# EEG gets most global guidance

GATE_INIT = {
    # instead of reintroducing a second hardcoded layer.
    "audio": 0.0075,
    "eeg":   0.75,
    "text":  0.35,
}

WEIGHT_CLIP_RANGE = (0.3, 1.5)

LR_CANDIDATES = [
    5e-4,
    2e-4,
    1e-4,
    5e-5,
    2e-5
]

# Auto-load previously-searched best config as the standing baseline
# hand-set defaults above instead.
FL_USE_SEARCHED_BASELINE = True
try:
    _best_path = f"{OUT}/fl_hparam_search_best.json"
    if FL_USE_SEARCHED_BASELINE and os.path.exists(_best_path):
        _b = json.load(open(_best_path)); _bc = _b.get("best", {})
        for _k in ("LR_HEAD", "COMPACT_LAMBDA", "PROTO_TEMPERATURE", "AUGMENT_NOISE_STD",
                   "HEAD_DROPOUT", "HIDDEN_DIM", "LOCAL_EPOCHS",
                   "CE_WEIGHT", "FOCAL_GAMMA"):
            if _k in _bc:
                globals()[_k] = int(_bc[_k]) if _k in ("HIDDEN_DIM", "LOCAL_EPOCHS") else _bc[_k]
        if "GATE_INIT" in _b:
            GATE_INIT = {k: float(v) for k, v in _b["GATE_INIT"].items()}
        print(f"[baseline] loaded searched config from {_best_path} "
              f"(val macro-F1={_bc.get('val_macro_f1','?')}) -> overriding the defaults above")
    else:
        print("[baseline] using hand-set defaults (no searched config found yet)")
except Exception as _e:
    print(f"[baseline] could not load searched config ({_e}); using hand-set defaults")

print(f"N_CLIENTS={N_CLIENTS}")
print(f"LOCAL_EPOCHS={LOCAL_EPOCHS}")
print(f"N_ROUNDS={N_ROUNDS}")
print(f"WARMUP_ROUNDS={WARMUP_ROUNDS}")
print(f"HIDDEN_DIM={HIDDEN_DIM}")
print(f"N_HEAD_LAYERS={N_HEAD_LAYERS}")
print(f"USE_CLIENT_RELIABILITY_WEIGHTING={USE_CLIENT_RELIABILITY_WEIGHTING}")
print(f"GATE_INIT={GATE_INIT}")



In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import math
import matplotlib.pyplot as plt


def _safe_normalize(x, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))

def make_simplex_etf(n_classes, dim, device=None, dtype=torch.float32):
    """
    Deterministic ETF / simplex initialization.
    For 3 classes this gives an equiangular triangle embedded in SHARED_DIM.
    """
    if n_classes < 2:
        raise ValueError("n_classes must be >= 2")
    if n_classes == 2:
        base = torch.tensor([[1.0], [-1.0]], dtype=dtype, device=device)
    elif n_classes == 3:
        base = torch.tensor([
            [1.0, 0.0],
            [-0.5,  math.sqrt(3) / 2.0],
            [-0.5, -math.sqrt(3) / 2.0],
        ], dtype=dtype, device=device)
    else:
        eye = torch.eye(n_classes, dtype=dtype, device=device)
        eye = eye - eye.mean(dim=0, keepdim=True)
        q, _ = torch.linalg.qr(eye.T, mode="reduced")
        base = q.T[:n_classes]
    if base.shape[1] < dim:
        pad = torch.zeros((n_classes, dim - base.shape[1]), dtype=dtype, device=device)
        base = torch.cat([base, pad], dim=1)
    elif base.shape[1] > dim:
        base = base[:, :dim]
    return _safe_normalize(base, dim=-1)

def _init_proto_bank(n_classes=N_CLASSES, dim=SHARED_DIM, use_etf=USE_ETF_INIT, device=None):
    if use_etf:
        return {c: make_simplex_etf(n_classes, dim, device=device)[c].detach().clone()
                for c in range(n_classes)}
    g = torch.randn(n_classes, dim, device=device)
    g = _safe_normalize(g, dim=-1)
    return {c: g[c].detach().clone() for c in range(n_classes)}

class ProjectionHead(nn.Module):
    """Native embedding dim -> shared 128-d. Configurable depth/width (HIDDEN_DIM, N_HEAD_LAYERS).
    Outputs are L2-normalized so prototypes live on the unit hypersphere."""
    def __init__(self, in_dim, out_dim=None, hidden_dim=None, n_layers=None, dropout=None):
        super().__init__()
        # time and would go stale inside a search loop).
        out_dim    = SHARED_DIM    if out_dim    is None else out_dim
        hidden_dim = HIDDEN_DIM    if hidden_dim is None else hidden_dim
        n_layers   = N_HEAD_LAYERS if n_layers   is None else n_layers
        dropout    = HEAD_DROPOUT  if dropout    is None else dropout
        layers = []
        d = in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout)]
            d = hidden_dim
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)
        self.identity_path = (in_dim == out_dim)
        if USE_CLASSIFIER_HEAD:
            self.classifier = nn.Sequential(
                nn.Linear(out_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, N_CLASSES))
        else:
            self.classifier = None

    def forward(self, x):
        out = self.net(x)
        if self.identity_path:
            out = 0.5 * x + 0.5 * out
        return F.normalize(out, dim=-1)

    def classify(self, emb):
        """Logits from the projected (already L2-normalized) representation."""
        return self.classifier(emb)

def compute_sample_size_weights(client_data, clip_range=WEIGHT_CLIP_RANGE):
    """Kept for ablation only. Main runs use adaptive client gates + class-wise reliability."""
    sizes = {silo: len(y) for silo, (X, y) in client_data.items()}
    total = sum(sizes.values())
    n_clients = len(sizes)
    raw = {silo: (total / (n_clients * n)) for silo, n in sizes.items()}
    lo, hi = clip_range
    return {silo: float(np.clip(w, lo, hi)) for silo, w in raw.items()}

def compute_local_prototypes(emb, y, n_classes=N_CLASSES):
    protos = {}
    for c in range(n_classes):
        mask = (y == c)
        n = int(mask.sum().item())
        if n > 0:
            protos[c] = (F.normalize(emb[mask].mean(dim=0).detach(), dim=-1), n)
    return protos

def aggregate_prototypes_naive(client_protos, n_classes=N_CLASSES):
    """Vanilla FedProto: weighted-average by sample count."""
    g = {}
    for c in range(n_classes):
        vecs, w = [], []
        for cp in client_protos:
            if c in cp:
                v, n = cp[c]
                vecs.append(F.normalize(v, dim=-1))
                w.append(n)
        if vecs:
            w = torch.tensor(w, dtype=torch.float32)
            w = w / w.sum().clamp_min(1e-12)
            g[c] = _safe_normalize((torch.stack(vecs) * w.unsqueeze(-1)).sum(dim=0), dim=-1)
    return g

def aggregate_prototypes_reliability(
    client_protos,
    client_reliability,
    n_classes=N_CLASSES,
    eps=1e-8
):
    """
    Reliability-weighted prototype aggregation.

    SIMPLIFIED (per literature-review report discussion): removed the hardcoded modality
    multipliers (audio 1.50x / text 1.25x / eeg 1.00x) that were stacked on top of the
    per-class reliability score. That was a second, redundant personalization layer on top
    of the adaptive client_gates mechanism already in run_one_round -- two overlapping
    hand-tuned mechanisms pulling the same direction makes the result hard to attribute and
    introduces unexplained constants a thesis reviewer would reasonably question. The adaptive
    gate (GATE_INIT + momentum adaptation) is now the SINGLE mechanism driving per-silo
    personalization; this function relies purely on the per-class reliability signal itself.
    """

    g = {}

    silo_names = list(client_reliability.keys())

    for c in range(n_classes):

        vecs = []
        weights = []

        for idx, cp in enumerate(client_protos):

            if c not in cp:
                continue

            proto, _ = cp[c]

            if idx < len(silo_names):
                silo_name = silo_names[idx]

                rel = float(
                    client_reliability
                    .get(silo_name, {})
                    .get(c, 0.0)
                )
            else:
                rel = 0.0

            rel = max(rel, eps)

            vecs.append(
                F.normalize(proto, dim=-1)
            )

            weights.append(rel)

        if len(vecs) == 0:
            continue

        weights = torch.tensor(
            weights,
            dtype=torch.float32,
            device=vecs[0].device
        )

        weights = (
            weights /
            weights.sum().clamp_min(eps)
        )

        g[c] = _safe_normalize(
            (
                torch.stack(vecs)
                * weights.unsqueeze(-1)
            ).sum(dim=0),
            dim=-1
        )

    return g

class TrainableGlobalPrototypes(nn.Module):
    """FedTGP-style trainable global prototypes, now ETF-initialized and kept on the hypersphere."""
    def __init__(self, n_classes=N_CLASSES, dim=SHARED_DIM, margin=0.5, init_prototypes=None):
        super().__init__()
        if init_prototypes is None:
            init_prototypes = _init_proto_bank(n_classes=n_classes, dim=dim, use_etf=True)
        init = torch.stack([init_prototypes[c] for c in range(n_classes)], dim=0)
        self.prototypes = nn.Parameter(F.normalize(init, dim=-1))
        self.margin = margin

    def contrastive_update_loss(self, client_protos):
        losses = []
        g = F.normalize(self.prototypes, dim=-1)
        for cp in client_protos:
            for c, (local_proto, _n) in cp.items():
                local_proto = F.normalize(local_proto, dim=-1)
                pos = F.cosine_similarity(local_proto.unsqueeze(0), g[c].unsqueeze(0))
                negs = torch.stack([F.cosine_similarity(local_proto.unsqueeze(0), g[k].unsqueeze(0))
                                    for k in range(g.shape[0]) if k != c])
                losses.append(F.relu(self.margin - (pos - negs.mean())))
        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=g.device, requires_grad=True)

    def as_dict(self):
        g = F.normalize(self.prototypes, dim=-1).detach()
        return {c: g[c] for c in range(g.shape[0])}

def fedproto_local_loss(emb, y, global_protos, temperature=None, class_weights=None):
    """Prototypical contrastive loss: pull toward own-class global prototype, push from others."""
    temperature = PROTO_TEMPERATURE if temperature is None else temperature
    if not global_protos:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)

    classes_present = sorted(global_protos.keys())
    if len(classes_present) < 2:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)

    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])
    losses = []
    for i in range(emb.shape[0]):
        yi = int(y[i].item())
        if yi not in global_protos:
            continue
        sims = F.cosine_similarity(F.normalize(emb[i].unsqueeze(0), dim=-1), proto_mat) / temperature
        target = classes_present.index(yi)
        ce = F.cross_entropy(sims.unsqueeze(0), torch.tensor([target], device=emb.device))
        if class_weights is not None:
            ce = ce * class_weights[yi]
        losses.append(ce)
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=emb.device, requires_grad=True)

def intra_class_compactness_loss(emb, y, temperature=0.07):
    emb = F.normalize(emb, dim=-1)
    sim = emb @ emb.T
    losses = []
    for i in range(len(y)):
        pos_mask = (y == y[i])
        pos_mask[i] = False
        neg_mask = (y != y[i])
        if pos_mask.sum() == 0:
            continue
        pos = sim[i][pos_mask] / temperature
        neg = sim[i][neg_mask] / temperature
        numerator = torch.exp(pos).sum()
        denominator = numerator + torch.exp(neg).sum()
        losses.append(-torch.log(numerator / denominator))
    if len(losses) == 0:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)
    return torch.stack(losses).mean()

def classification_loss(logits, y, class_weights=None):
    """Class-balanced classifier loss. Focal (default) emphasizes hard / minority examples,
    which is what macro-F1 rewards on these imbalanced silos; falls back to weighted CE with
    label smoothing when USE_FOCAL_LOSS is False."""
    if USE_FOCAL_LOSS:
        logp = F.log_softmax(logits, dim=-1)
        ce = F.nll_loss(logp, y, weight=class_weights, reduction='none')
        pt = logp.gather(1, y.unsqueeze(1)).squeeze(1).exp()
        return (((1.0 - pt) ** FOCAL_GAMMA) * ce).mean()
    return F.cross_entropy(logits, y, weight=class_weights, label_smoothing=CE_LABEL_SMOOTHING)

def _mixup_ce(head, xb, yb, class_weights=None):
    """Mixup on the classification path only (proto/compact losses keep hard labels)."""
    if (not USE_MIXUP) or MIXUP_ALPHA <= 0 or xb.shape[0] < 2:
        return classification_loss(head.classify(head(xb)), yb, class_weights)
    lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
    perm = torch.randperm(xb.shape[0], device=xb.device)
    xmix = lam * xb + (1.0 - lam) * xb[perm]
    logits = head.classify(head(xmix))
    return (lam * classification_loss(logits, yb, class_weights)
            + (1.0 - lam) * classification_loss(logits, yb[perm], class_weights))

def positive_prob(head, emb):
    """P(class=1) from the classifier head on a projected (normalized) embedding."""
    return torch.softmax(head.classify(emb), dim=-1)[:, 1]

def best_threshold_on_val(head, Xval, yval):
    """Binary only: choose the decision threshold that maximizes macro-F1 on this silo's VAL
    set. Returns 0.5 for the multi-class task or when tuning is disabled."""
    if N_CLASSES != 2 or not USE_THRESHOLD_TUNING \
       or PREDICT_WITH != 'classifier' or getattr(head, 'classifier', None) is None:
        return 0.5
    head.eval()
    with torch.no_grad():
        p = positive_prob(head, head(Xval)).cpu().numpy()
    yv = yval.cpu().numpy() if torch.is_tensor(yval) else np.asarray(yval)
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, THRESHOLD_GRID):
        f1 = f1_score(yv, (p >= t).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t

def predict_labels(head, emb, global_protos, pos_threshold=None):
    """Unified prediction: trained classifier head if enabled, else nearest-prototype.
    For binary with a tuned pos_threshold, decide by P(class=1) >= threshold."""
    if PREDICT_WITH == 'classifier' and getattr(head, 'classifier', None) is not None:
        if N_CLASSES == 2 and pos_threshold is not None:
            return (positive_prob(head, emb) >= pos_threshold).long()
        return head.classify(emb).argmax(dim=-1)
    return predict_via_prototypes(emb, global_protos)

def predict_via_prototypes(emb, global_protos):
    classes_present = sorted(global_protos.keys())
    if not classes_present:
        return torch.zeros(len(emb), dtype=torch.long, device=emb.device)
    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])
    sims = F.normalize(emb, dim=-1) @ F.normalize(proto_mat, dim=-1).T
    idx = sims.argmax(dim=-1)
    return torch.tensor([classes_present[i] for i in idx.tolist()], device=emb.device)

def per_class_reliability(y_true, y_pred, n_classes=N_CLASSES):
    scores = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=list(range(n_classes)),
        zero_division=0
    )
    return {c: float(scores[c]) for c in range(n_classes)}

def class_weights_from_labels(y, n_classes=N_CLASSES, eps=1e-6):
    """Inverse-frequency class weights (present classes only; absent -> 0).
    Normalized so the mean weight over present classes is ~1."""
    counts = torch.bincount(y, minlength=n_classes).float()
    present = counts > 0
    w = torch.zeros(n_classes)
    w[present] = 1.0 / counts[present]
    w[present] = w[present] * (present.sum().float() / w[present].sum().clamp_min(eps))
    return w

def run_one_round(
    tdata,
    heads,
    global_protos,
    optimizers,
    local_epochs=None,
    trainable_global=None,
    client_gates=None,
    warmup=False,
    use_reliability_weighting=None,
    grad_clip=None
):
    # live-resolve so a search loop can vary these via globals
    local_epochs = LOCAL_EPOCHS if local_epochs is None else local_epochs
    use_reliability_weighting = USE_CLIENT_RELIABILITY_WEIGHTING if use_reliability_weighting is None else use_reliability_weighting
    grad_clip = GRAD_CLIP if grad_clip is None else grad_clip

    client_local_protos = []
    client_reliability = {}
    round_losses = []

    for silo in tdata:

        Xtr, ytr = tdata[silo]["train"]
        Xva, yva = tdata[silo]["val"]

        head = heads[silo]
        opt = optimizers[silo]

        gate = (
            client_gates.get(silo, 0.5)
            if client_gates is not None
            else 0.5
        )

        head.train()
        cw = class_weights_from_labels(ytr) if USE_CLASS_WEIGHTING else None
        n_train = Xtr.shape[0]

        for _ in range(local_epochs):

            perm = torch.randperm(n_train)

            for start in range(0, n_train, BATCH_SIZE):

                idx = perm[start:start + BATCH_SIZE]
                xb = Xtr[idx]
                yb = ytr[idx]

                # cheap augmentation: jitter embeddings to expand effective sample size
                if AUGMENT_NOISE_STD > 0:
                    xb = xb + AUGMENT_NOISE_STD * torch.randn_like(xb)

                opt.zero_grad()

                emb = head(xb)

                proto_loss = (
                    fedproto_local_loss(emb, yb, global_protos, class_weights=cw)
                    if len(global_protos) > 0
                    else torch.tensor(0.0, device=emb.device, requires_grad=True)
                )

                if USE_COMPACTNESS:
                    compact_loss = intra_class_compactness_loss(emb, yb)
                else:
                    compact_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)

                # FedPer-style supervised classification term (the headline predictor).
                if USE_CLASSIFIER_HEAD and head.classifier is not None:
                    ce_loss = _mixup_ce(head, xb, yb, class_weights=cw)
                else:
                    ce_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)

                total_loss = gate * proto_loss + COMPACT_LAMBDA * compact_loss + CE_WEIGHT * ce_loss

                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)

                opt.step()

        head.eval()

        with torch.no_grad():

            emb_train = head(Xtr)

            local_proto = compute_local_prototypes(
                emb_train,
                ytr
            )

            client_local_protos.append(
                local_proto
            )

            emb_val = head(Xva)

            pred_val = predict_via_prototypes(
                emb_val,
                global_protos
            )

            client_reliability[silo] = (
                per_class_reliability(
                    yva.cpu().numpy(),
                    pred_val.cpu().numpy(),
                    N_CLASSES
                )
            )

        round_losses.append(
            float(total_loss.item())
        )

    # Global prototype update
    if trainable_global is not None:

        proto_optimizer = torch.optim.Adam(
            trainable_global.parameters(),
            lr=1e-3
        )

        proto_optimizer.zero_grad()

        gp_loss = (
            trainable_global
            .contrastive_update_loss(
                client_local_protos
            )
        )

        gp_loss.backward()

        proto_optimizer.step()

        with torch.no_grad():
            trainable_global.prototypes.data = (
                F.normalize(
                    trainable_global.prototypes.data,
                    dim=-1
                )
            )

        new_global = (
            trainable_global.as_dict()
        )

    else:

        if (
            use_reliability_weighting
            and len(client_reliability) > 0
        ):

            new_global = (
                aggregate_prototypes_reliability(
                    client_local_protos,
                    client_reliability
                )
            )

        else:

            new_global = (
                aggregate_prototypes_naive(
                    client_local_protos
                )
            )

    return (
        new_global,
        client_local_protos,
        float(np.mean(round_losses)),
        client_reliability
    )

print("Core FL functions defined (ETF init + warm-up + reliability-weighted aggregation enabled).")

In [ ]:
labels = pd.read_csv(LABELS_CSV)
labels["subject_id"] = labels["subject_id"].astype(str)

def load_eeg():
    """Concatenate the three locked-split files; ids are MODMA subject_id strings e.g. '2010001'."""
    Xs, ids, ys = [], [], []
    for split in ("train", "val", "test"):
        Xs.append(np.load(f"{EEG_EMBED_DIR}/eeg_embeddings_{split}.npy"))
        ids.append(np.load(f"{EEG_EMBED_DIR}/eeg_subj_ids_{split}.npy", allow_pickle=True))
    X = np.concatenate(Xs, axis=0)
    ids = np.concatenate(ids, axis=0)
    return X, ids

def load_text():
    d = np.load(TEXT_EMBED_NPZ, allow_pickle=True)
    return d["X"], d["ids"]

def load_audio():
    """Reconstruct the LOCKED 1536-dim (layer-10 mean+std) input from raw per-participant turn
    embeddings. Verified numerically identical to the original DAIC notebook's pool_mean_std()
    + layer-slice -- this is NOT an approximation, it reproduces the exact locked-model input."""
    ids, X = [], []
    for f in sorted(glob.glob(f"{AUDIO_TURN_DIR}/*.npz")):
        pid = int(os.path.basename(f).split(".")[0])
        d = np.load(f)
        emb = d["emb"]                                  # (n_turns, 13, 768)
        layer_emb = emb[:, AUDIO_LOCK_LAYER, :]          # (n_turns, 768)
        mu = layer_emb.mean(axis=0)
        sd = layer_emb.std(axis=0)
        X.append(np.concatenate([mu, sd]))               # (1536,)
        ids.append(pid)
    return np.stack(X).astype(np.float32), np.array(ids)

print("Loading EEG (confirmed 128-dim, n=55), TEXT (confirmed 768-dim, n=99),")
print("AUDIO (reconstructed 1536-dim from turn embeddings, n~189; this matches the LOCKED SVM input)...\n")
try:
    X_eeg_raw, ids_eeg = load_eeg();  print("EEG  :", X_eeg_raw.shape, "| dtype ids:", ids_eeg.dtype)
except Exception as e: print("EEG load failed:", e)
try:
    X_txt_raw, ids_txt = load_text(); print("Text :", X_txt_raw.shape, "| dtype ids:", ids_txt.dtype)
except Exception as e: print("Text load failed:", e)
try:
    X_aud_raw, ids_aud = load_audio(); print("Audio:", X_aud_raw.shape, "| dtype ids:", ids_aud.dtype,
                                              "\n   (NOTE: this is RECONSTRUCTED from turn_embeddings/*.npz,")
    print("    NOT loaded from audio_embeddings_*.npy or X_audio.npy -- those are different, non-locked tracks.)")
except Exception as e: print("Audio load failed:", e)

In [ ]:
def join_silo(silo_name, X_raw, ids_raw):
    """Join an embedding array to harmonized labels by subject_id, with a loud diagnostic if the
    join rate is suspiciously low (catches silent id-format mismatches, e.g. zero-padding or
    str-vs-int, before they corrupt the silo's training set)."""
    sub = labels[labels.silo == silo_name].copy()
    sub["subject_id"] = sub["subject_id"].astype(str)
    id_to_row = {str(i): r for r, i in enumerate(ids_raw)}
    keep_rows, keep_y = [], []
    for _, row in sub.iterrows():
        sid = row.subject_id
        if sid in id_to_row and not pd.isna(row.y_unified):
            keep_rows.append(id_to_row[sid]); keep_y.append(int(row.y_unified))
    X = X_raw[keep_rows]
    y = np.array(keep_y, dtype=np.int64)
    if TASK == "binary":
        _cut = BINARY_POSITIVE_FROM[silo_name] if isinstance(BINARY_POSITIVE_FROM, dict) else BINARY_POSITIVE_FROM
        y = (y >= _cut).astype(np.int64)   # per-silo positive cut
    join_rate = len(y) / max(len(sub), 1)
    print(f"  [{silo_name}] joined {len(y)}/{len(sub)} subjects "
          f"(dropped = no matching embedding id or missing label)")
    if join_rate < 0.9:
        print(f"  *** WARNING: only {join_rate:.0%} of {silo_name} subjects joined. ***")
        print(f"      sample harmonized ids : {list(sub.subject_id.head(3))}")
        print(f"      sample embedding ids  : {[str(i) for i in ids_raw[:3]]}")
        print(f"      -> check id format (zero-padding, str-vs-int, leading characters) before trusting results.")
    return X, y

X_eeg, y_eeg = join_silo("eeg",   X_eeg_raw, ids_eeg)
X_txt, y_txt = join_silo("text",  X_txt_raw, ids_txt)
X_aud, y_aud = join_silo("audio", X_aud_raw, ids_aud)

from sklearn.model_selection import train_test_split
def subject_split(X, y, val_frac=0.2, test_frac=0.2, seed=42):
    """Subject-level (no-leakage) train/val/test split, stratified where class counts allow."""
    n = len(y)
    idx = np.arange(n)
    try:
        idx_tr, idx_rest = train_test_split(idx, test_size=val_frac+test_frac, random_state=seed, stratify=y)
        rel_test = test_frac/(val_frac+test_frac)
        idx_val, idx_te = train_test_split(idx_rest, test_size=rel_test, random_state=seed, stratify=y[idx_rest])
    except ValueError:
        print(f"    WARNING: a class has too few subjects to stratify -- using unstratified split")
        idx_tr, idx_rest = train_test_split(idx, test_size=val_frac+test_frac, random_state=seed)
        rel_test = test_frac/(val_frac+test_frac)
        idx_val, idx_te = train_test_split(idx_rest, test_size=rel_test, random_state=seed)
    return (X[idx_tr], y[idx_tr]), (X[idx_val], y[idx_val]), (X[idx_te], y[idx_te])

splits = {
    "eeg":   subject_split(X_eeg, y_eeg),
    "text":  subject_split(X_txt, y_txt),
    "audio": subject_split(X_aud, y_aud),
}
for silo, (tr, va, te) in splits.items():
    if TASK == "binary":
        cut = BINARY_POSITIVE_FROM[silo] if isinstance(BINARY_POSITIVE_FROM, dict) else BINARY_POSITIVE_FROM
        cut_str = f" | binary cut: y>={cut}"
    else:
        cut_str = ""
    print(f"{silo:6s} train={len(tr[1])} val={len(va[1])} test={len(te[1])} "
          f"| train class counts={np.bincount(tr[1], minlength=N_CLASSES).tolist()}" + cut_str)

## Stage 1a — Baseline: per-silo standalone + decision-level late-fusion ensemble

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def fit_probe(X_tr, y_tr, seed):
    sc = StandardScaler().fit(X_tr)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed).fit(sc.transform(X_tr), y_tr)
    return sc, clf

def eval_probe(sc, clf, X, y):
    pred = clf.predict(sc.transform(X))
    return f1_score(y, pred, average="macro", zero_division=0), pred

standalone_results = {}
for silo, (tr, va, te) in splits.items():
    f1s = []
    for seed in range(N_SEEDS):
        sc, clf = fit_probe(*tr, seed=seed)
        f1, _ = eval_probe(sc, clf, *te)
        f1s.append(f1)
    standalone_results[silo] = (np.mean(f1s), np.std(f1s))
    print(f"[standalone] {silo:6s} test macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}  (n_seeds={N_SEEDS})")

In [ ]:
# not features -- no patient needs to be present in more than one silo.
from sklearn.linear_model import LogisticRegression as LR

def fit_proba_probe(X_tr, y_tr, seed, n_classes=N_CLASSES):
    sc = StandardScaler().fit(X_tr)
    clf = LR(max_iter=2000, class_weight="balanced", random_state=seed)
    clf.fit(sc.transform(X_tr), y_tr)
    return sc, clf

def ensemble_eval_on_test_union(seed):
    """Evaluate the ensemble idea the only way that's meaningful under no-overlap:
    each silo's OWN test subjects get classified by (a) their own probe and (b) what
    an averaged-confidence ensemble WOULD look like if multiple silos saw the same
    patient. Since they never do here, we report the ensemble as a method comparison
    on a *common protocol* (same probe family, same CV scheme) rather than literally
    averaging probabilities for the same patient (which is impossible under no-overlap).
    This cell mainly documents WHY decision-level literal fusion needs paired patients,
    and falls back to reporting standalone-best as the practical "late fusion" floor here."""
    pass

print("NOTE: literal decision-level probability averaging requires the SAME patient to be")
print("seen by >=2 silos, which cannot happen under the no-overlap constraint. The practical")
print("'late-fusion' floor in this setting is therefore: best standalone silo performance.")
best_silo = max(standalone_results, key=lambda s: standalone_results[s][0])
print(f"\n>>> FLOOR TO BEAT: best standalone silo = '{best_silo}' "
      f"({standalone_results[best_silo][0]:.3f} +/- {standalone_results[best_silo][1]:.3f} macro-F1)")
print(">>> Cross-silo prototype FL must beat this on each silo's OWN test set to demonstrate transfer.")

## Stage 1b — FedProto (naive) vs. FedTGP-style trainable global prototypes

In [ ]:
def to_tensor_data(splits, standardize=USE_INPUT_STANDARDIZATION):
    """Per-silo z-scoring of head inputs using TRAIN stats only (no leakage).
    The standalone LR baseline already standardizes its inputs; the FL heads did not,
    which handicapped them on raw-scale embeddings. This puts both on equal footing."""
    out = {}
    for silo, (tr, va, te) in splits.items():
        Xtr, ytr = tr; Xva, yva = va; Xte, yte = te
        if standardize:
            mu = Xtr.mean(axis=0, keepdims=True)
            sd = Xtr.std(axis=0, keepdims=True) + 1e-6
            Xtr = (Xtr - mu) / sd; Xva = (Xva - mu) / sd; Xte = (Xte - mu) / sd
        out[silo] = {
            "train": (torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long)),
            "val":   (torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva, dtype=torch.long)),
            "test":  (torch.tensor(Xte, dtype=torch.float32), torch.tensor(yte, dtype=torch.long)),
        }
    return out

tdata = to_tensor_data(splits)
in_dims = {silo: tdata[silo]["train"][0].shape[1] for silo in tdata}
print("input dims per client:", in_dims)

def per_silo_val_macro_f1(heads, global_protos, tdata):
    """Per-silo VAL macro-F1 dict (no leakage; never touches test). Uses the unified
    predictor (trained classifier head if enabled, else nearest-prototype)."""
    out = {}
    for silo in tdata:
        heads[silo].eval()
        with torch.no_grad():
            emb = heads[silo](tdata[silo]["val"][0])
            pred = predict_labels(heads[silo], emb, global_protos)
        out[silo] = f1_score(tdata[silo]["val"][1].numpy(), pred.cpu().numpy(),
                             average="macro", zero_division=0)
    return out

def pooled_val_macro_f1(heads, global_protos, tdata):
    """Mean per-silo VAL macro-F1 (used for early stopping)."""
    d = per_silo_val_macro_f1(heads, global_protos, tdata)
    return float(np.mean(list(d.values())))

def run_fl_training(
    tdata,
    in_dims,
    seed,
    use_trainable_global=False,
    n_rounds=N_ROUNDS,
    patience=EARLY_STOP_PATIENCE,
    lr=None,
    use_sample_size_weighting=USE_SAMPLE_SIZE_WEIGHTING
):

    torch.manual_seed(seed)

    lr = lr if lr is not None else LR_HEAD

    heads = {
        silo: ProjectionHead(in_dims[silo])
        for silo in tdata
    }

    optimizers = {
        silo: torch.optim.AdamW(
            h.parameters(),
            lr=lr,
            weight_decay=1e-4
        )
        for silo, h in heads.items()
    }

    trainable_global = (
        TrainableGlobalPrototypes(
            n_classes=N_CLASSES,
            dim=SHARED_DIM,
            init_prototypes=_init_proto_bank(
                N_CLASSES,
                SHARED_DIM,
                use_etf=USE_ETF_INIT
            )
        )
        if use_trainable_global
        else None
    )

    global_protos = (
        trainable_global.as_dict()
        if trainable_global is not None
        else _init_proto_bank(
            N_CLASSES,
            SHARED_DIM,
            use_etf=USE_ETF_INIT
        )
    )

    client_data = {
        silo: tdata[silo]["train"]
        for silo in tdata
    }

    loss_weights = (
        compute_sample_size_weights(client_data)
        if use_sample_size_weighting
        else None
    )

    client_gates = dict(GATE_INIT)

    best_val = -1.0
    best_round = -1
    best_state = None

    best_protos = {
        c: v.clone()
        for c, v in global_protos.items()
    }

    no_improve = 0
    history = []

    for r in range(n_rounds):

        warmup = (r < WARMUP_ROUNDS)

        global_protos, _, loss, client_reliability = run_one_round(
            tdata=tdata,
            heads=heads,
            global_protos=global_protos,
            optimizers=optimizers,
            trainable_global=trainable_global,
            client_gates=client_gates,
            warmup=warmup,
            use_reliability_weighting=USE_CLIENT_RELIABILITY_WEIGHTING
        )

        # Improved gate adaptation
        if USE_CLIENT_RELIABILITY_WEIGHTING:

            for silo in client_gates:

                rel_vals = list(
                    client_reliability.get(
                        silo,
                        {}
                    ).values()
                )

                rel = (
                    float(np.mean(rel_vals))
                    if len(rel_vals) > 0
                    else 0.5
                )

                rel = float(
                    np.clip(
                        rel,
                        GATE_MIN,
                        GATE_MAX
                    )
                )

                if silo == "audio":

                    target_gate = min(
                        rel * 0.01,
                        0.0075
                    )

                # Text moderately local
                elif silo == "text":

                    target_gate = min(
                        rel * 0.60,
                        0.50
                    )

                # EEG gets most global guidance
                else:

                    target_gate = rel

                client_gates[silo] = (
                    GATE_ADAPT_MOMENTUM
                    * client_gates[silo]
                    +
                    (1.0 - GATE_ADAPT_MOMENTUM)
                    * target_gate
                )

        val_f1 = pooled_val_macro_f1(
            heads,
            global_protos,
            tdata
        )

        history.append({
            "round": r,
            "loss": loss,
            "val_macro_f1": val_f1,
            "warmup": warmup,
            **{
                f"gate_{s}": float(client_gates[s])
                for s in client_gates
            }
        })

        if val_f1 > best_val:

            best_val = val_f1
            best_round = r

            best_state = {
                silo: {
                    k: v.clone()
                    for k, v in h.state_dict().items()
                }
                for silo, h in heads.items()
            }

            best_protos = {
                c: v.clone()
                for c, v in global_protos.items()
            }

            no_improve = 0

        else:
            no_improve += 1

        if no_improve >= patience:
            break

    for silo, h in heads.items():
        h.load_state_dict(
            best_state[silo]
        )

    return (
        heads,
        best_protos,
        best_round,
        pd.DataFrame(history)
    )
print("FL training loop defined: ETF init, warm-up, adaptive gates, and early stopping on pooled val macro-F1.")


In [ ]:
import itertools, random, time, json as _json
from sklearn.model_selection import train_test_split as _tts

# ---- search controls ----
SEARCH_SELECT_ON   = "vanilla"   # "vanilla" | "tgp" | "both": which variant's VAL macro-F1 to optimize
SEARCH_OBJECTIVE   = "min"       # "mean" | "min" | "weak2": reduce per-silo VAL to one score.
                                 #   "min"  -> maximize the WORST silo (pushes audio+text up,
                                 #             the right target for 'all three > 0.5').
                                 #   "weak2"-> mean of the two weakest silos (text+audio here).
SEARCH_SEEDS       = 2           # model-init seeds averaged per trial (variance control)
SEARCH_ROUNDS      = 25          # rounds per trial (early-stopped); < full N_ROUNDS for speed
SEARCH_PATIENCE    = 10
SEARCH_MAX_TRIALS  = 30          # budget; a larger grid is sampled down to this many
SEARCH_STRATEGY    = "random"    # "random" | "grid"
SEARCH_RANDOM_SEED = 0

SEARCH_VAL_REPEATS     = 1
SEARCH_VAL_SPLIT_SEEDS = [2024, 7, 99, 1234, 555, 31, 808]

SEARCH_SPACE = {
    "LR_HEAD":           [5e-4, 2e-4, 1e-4, 5e-5],
    "COMPACT_LAMBDA":    [0.0, 0.1, 0.2, 0.4],
    "PROTO_TEMPERATURE": [0.05, 0.1, 0.2],
    "AUGMENT_NOISE_STD": [0.0, 0.05, 0.1],
    "HEAD_DROPOUT":      [0.1, 0.2, 0.3],
    "HIDDEN_DIM":        [128, 256],
    "LOCAL_EPOCHS":      [5, 10],
    "GATE_EEG":          [0.5, 0.75],   # eeg global-prototype pull (init)
    "GATE_TEXT":         [0.2, 0.35, 0.5],
    # classifier-head levers (only matter when USE_CLASSIFIER_HEAD=True):
    "CE_WEIGHT":         [0.5, 1.0, 2.0],
    "FOCAL_GAMMA":       [0.0, 1.0, 2.0],
}

_GATE_KEYS = {"GATE_EEG": "eeg", "GATE_TEXT": "text"}  # audio stays at its configured small value

def _apply_cfg(cfg):
    """Write a trial config into the module globals so the patched core picks it up."""
    g = globals()
    for k, v in cfg.items():
        if k in _GATE_KEYS:
            gi = dict(g["GATE_INIT"]); gi[_GATE_KEYS[k]] = float(v); g["GATE_INIT"] = gi
        else:
            g[k] = v

def _resampled_tdata(split_seed):
    """Rebuild tdata by re-splitting ONLY the non-test pool (train+val) per silo into a fresh
    train/val; the canonical TEST set is kept frozen -> no test leakage during selection.
    Standardization stats are recomputed from each fold's own train (no leakage)."""
    new_splits = {}
    for silo, (tr, va, te) in splits.items():
        Xtr, ytr = tr; Xva, yva = va
        poolX = np.concatenate([Xtr, Xva], axis=0)
        poolY = np.concatenate([ytr, yva], axis=0)
        idx = np.arange(len(poolY))
        try:
            i_tr, i_va = _tts(idx, test_size=0.25, random_state=split_seed, stratify=poolY)
        except ValueError:
            i_tr, i_va = _tts(idx, test_size=0.25, random_state=split_seed)
        new_splits[silo] = ((poolX[i_tr], poolY[i_tr]),
                            (poolX[i_va], poolY[i_va]),
                            te)  # frozen canonical test (never read during search)
    return to_tensor_data(new_splits)

def _reduce_objective(per_silo):
    """Collapse a per-silo VAL macro-F1 dict to the scalar the search maximizes."""
    vals = list(per_silo.values())
    if SEARCH_OBJECTIVE == "min":
        return float(np.min(vals))
    if SEARCH_OBJECTIVE == "weak2":
        return float(np.mean(sorted(vals)[:2]))
    return float(np.mean(vals))

def _val_for_variant(use_tgp, lr, tdata_fold):
    """Average per-silo VAL macro-F1 across seeds, then reduce by SEARCH_OBJECTIVE."""
    acc = {silo: [] for silo in tdata_fold}
    for seed in range(SEARCH_SEEDS):
        heads, protos, _br, _h = run_fl_training(
            tdata_fold, in_dims, seed,
            use_trainable_global=use_tgp,
            n_rounds=SEARCH_ROUNDS, patience=SEARCH_PATIENCE,
            lr=lr, use_sample_size_weighting=USE_SAMPLE_SIZE_WEIGHTING)
        d = per_silo_val_macro_f1(heads, protos, tdata_fold)
        for s in d: acc[s].append(d[s])
    mean_per_silo = {s: float(np.mean(v)) for s, v in acc.items()}
    return _reduce_objective(mean_per_silo), mean_per_silo

def _trial_val(cfg, folds):
    _apply_cfg(cfg)
    lr = cfg["LR_HEAD"]
    scores = []
    silo_acc = {}
    for fold in folds:
        if SEARCH_SELECT_ON == "vanilla":
            sc, ps = _val_for_variant(False, lr, fold)
        elif SEARCH_SELECT_ON == "tgp":
            sc, ps = _val_for_variant(True, lr, fold)
        else:
            sc0, ps0 = _val_for_variant(False, lr, fold)
            sc1, ps1 = _val_for_variant(True, lr, fold)
            sc = 0.5 * (sc0 + sc1)
            ps = {s: 0.5 * (ps0[s] + ps1[s]) for s in ps0}
        scores.append(sc)
        for s in ps: silo_acc.setdefault(s, []).append(ps[s])
    per_silo = {s: float(np.mean(v)) for s, v in silo_acc.items()}
    return float(np.mean(scores)), per_silo

if SEARCH_VAL_REPEATS <= 1:
    _VAL_FOLDS = [tdata]
    print("Validation: 1 canonical split.")
else:
    _seeds = SEARCH_VAL_SPLIT_SEEDS[:SEARCH_VAL_REPEATS]
    _VAL_FOLDS = [_resampled_tdata(s) for s in _seeds]
    print(f"Validation: {len(_VAL_FOLDS)} resampled train/val folds (seeds {_seeds}); "
          f"canonical TEST frozen -> no leakage.")

# ---- build candidate list ----
keys = list(SEARCH_SPACE.keys())
grid = [dict(zip(keys, combo)) for combo in itertools.product(*[SEARCH_SPACE[k] for k in keys])]
rng = random.Random(SEARCH_RANDOM_SEED)
if SEARCH_STRATEGY == "random":
    rng.shuffle(grid)
candidates = grid[:SEARCH_MAX_TRIALS]
_fits = SEARCH_SEEDS * len(_VAL_FOLDS) * (2 if SEARCH_SELECT_ON == "both" else 1)
print(f"Search: {len(grid)} configs in grid -> evaluating {len(candidates)} "
      f"({SEARCH_STRATEGY}, select on '{SEARCH_SELECT_ON}', {_fits} model-fits/trial)\n")

_searchable = set(keys) | {"GATE_INIT"}
_orig = {k: (dict(globals()[k]) if k == "GATE_INIT" else globals().get(k)) for k in _searchable}

trials, best = [], None
t0 = time.time()
for i, cfg in enumerate(candidates):
    try:
        val, per_silo = _trial_val(cfg, _VAL_FOLDS)
    except Exception as e:
        print(f"  trial {i:02d} FAILED ({type(e).__name__}: {e}) -- skipped"); continue
    row = dict(cfg); row["val_score"] = val
    for s in per_silo: row[f"val_{s}"] = per_silo[s]
    trials.append(row)
    flag = ""
    if best is None or val > best["val_macro_f1"]:
        best = dict(cfg); best["val_macro_f1"] = val; best["per_silo"] = per_silo; flag = "  <-- new best"
    ps_str = " ".join(f"{s}={per_silo[s]:.2f}" for s in per_silo)
    print(f"  trial {i:02d}  obj({SEARCH_OBJECTIVE})={val:.4f}  [{ps_str}]  "
          + " ".join(f"{k}={cfg[k]}" for k in keys) + flag)

print(f"\nSearched {len(trials)} configs in {time.time()-t0:.0f}s.")
if best is None:
    print("No successful trial; restoring original config."); _apply_cfg({k: _orig[k] for k in keys})
else:
    # lock the winning config into the globals for the downstream full runs
    for k in keys:
        if k in _GATE_KEYS:
            _apply_cfg({k: best[k]})
        else:
            globals()[k] = best[k]
    LR_HEAD = best["LR_HEAD"]
    print(f">>> BEST objective({SEARCH_OBJECTIVE}) VAL = {best['val_macro_f1']:.4f} "
          f"(over {len(_VAL_FOLDS)} val fold(s) x {SEARCH_SEEDS} seeds)")
    print("    best per-silo VAL macro-F1: "
          + ", ".join(f"{s}={best['per_silo'][s]:.3f}" for s in best['per_silo']))
    print(">>> Locked config (used by the vanilla + FedTGP full runs below):")
    for k in keys:
        print(f"      {k:18s} = {best[k]}")
    print(f"      GATE_INIT          = {GATE_INIT}")
    trials_df = pd.DataFrame(trials).sort_values("val_score", ascending=False)
    trials_df.to_csv(f"{OUT}/fl_hparam_search_trials.csv", index=False)
    _json.dump({"select_on": SEARCH_SELECT_ON, "seeds": SEARCH_SEEDS, "rounds": SEARCH_ROUNDS,
                "val_repeats": SEARCH_VAL_REPEATS, "n_trials": len(trials),
                "best": best, "GATE_INIT": GATE_INIT},
               open(f"{OUT}/fl_hparam_search_best.json", "w"), indent=2, default=str)
    print(f"\nsaved: {OUT}/fl_hparam_search_trials.csv and {OUT}/fl_hparam_search_best.json")
    print("This best config is now the standing baseline: the config cell auto-loads it on the")
    print("next run (FL_USE_SEARCHED_BASELINE=True). Selection used VALIDATION only; the test")
    print("macro-F1 below is computed ONCE with this locked config -- no test-set selection.")


In [ ]:
print(f"Using LR_HEAD={LR_HEAD} (from sweep), sample-size weighting={USE_SAMPLE_SIZE_WEIGHTING}, client reliability weighting={USE_CLIENT_RELIABILITY_WEIGHTING}, grad_clip={GRAD_CLIP}, compactness={USE_COMPACTNESS} (lambda={COMPACT_LAMBDA})\n")
vanilla_test_f1 = {silo: [] for silo in tdata}
vanilla_perclass = {silo: [] for silo in tdata}
vanilla_rounds_locked = []
vanilla_hist_seed0 = None   # captured for the training-curve plot

for seed in range(N_SEEDS):
    heads, protos, best_round, hist = run_fl_training(tdata, in_dims, seed, use_trainable_global=False, lr=LR_HEAD)
    if seed == 0:
        vanilla_hist_seed0 = hist
    vanilla_rounds_locked.append(best_round)
    for silo in tdata:
        heads[silo].eval()
        thr = best_threshold_on_val(heads[silo], tdata[silo]["val"][0], tdata[silo]["val"][1])
        with torch.no_grad():
            emb = heads[silo](tdata[silo]["test"][0])
        pred = predict_labels(heads[silo], emb, protos, pos_threshold=thr)
        f1 = f1_score(tdata[silo]["test"][1].numpy(), pred.numpy(), average="macro", zero_division=0)
        pcf1 = f1_score(tdata[silo]["test"][1].numpy(), pred.numpy(),
                        average=None, labels=list(range(N_CLASSES)), zero_division=0)
        vanilla_perclass[silo].append(pcf1)
        vanilla_test_f1[silo].append(f1)
    print(f"  seed {seed}: locked at round {best_round}, "
          f"test macro-F1 -> " + ", ".join(f"{s}={vanilla_test_f1[s][-1]:.3f}" for s in tdata))

print(f"\nVanilla FedProto -- rounds locked across seeds: {vanilla_rounds_locked} "
      f"(mean={np.mean(vanilla_rounds_locked):.1f})")
for silo in tdata:
    m, s = np.mean(vanilla_test_f1[silo]), np.std(vanilla_test_f1[silo])
    print(f"  [vanilla FedProto] {silo:6s} test macro-F1: {m:.3f} +/- {s:.3f}")

In [ ]:
tgp_test_f1 = {silo: [] for silo in tdata}
tgp_rounds_locked = []
tgp_hist_seed0 = None   # captured for the training-curve plot
tgp_perclass = {silo: [] for silo in tdata}

for seed in range(N_SEEDS):
    heads, protos, best_round, hist = run_fl_training(tdata, in_dims, seed, use_trainable_global=True, lr=LR_HEAD)
    if seed == 0:
        tgp_hist_seed0 = hist
    tgp_rounds_locked.append(best_round)
    for silo in tdata:
        heads[silo].eval()
        thr = best_threshold_on_val(heads[silo], tdata[silo]["val"][0], tdata[silo]["val"][1])
        with torch.no_grad():
            emb = heads[silo](tdata[silo]["test"][0])
            pred = predict_labels(heads[silo], emb, protos, pos_threshold=thr)
        f1 = f1_score(tdata[silo]["test"][1].numpy(), pred.numpy(), average="macro", zero_division=0)
        pcf1 = f1_score(tdata[silo]["test"][1].numpy(), pred.numpy(),
                        average=None, labels=list(range(N_CLASSES)), zero_division=0)   # ADD
        tgp_perclass[silo].append(pcf1)
        tgp_test_f1[silo].append(f1)
    print(f"  seed {seed}: locked at round {best_round}, "
          f"test macro-F1 -> " + ", ".join(f"{s}={tgp_test_f1[s][-1]:.3f}" for s in tdata))

print(f"\nFedTGP-style -- rounds locked across seeds: {tgp_rounds_locked} "
      f"(mean={np.mean(tgp_rounds_locked):.1f})")
for silo in tdata:
    m, s = np.mean(tgp_test_f1[silo]), np.std(tgp_test_f1[silo])
    print(f"  [FedTGP-style] {silo:6s} test macro-F1: {m:.3f} +/- {s:.3f}")

In [ ]:
local_clf_test_f1 = {s: [] for s in tdata}
if not USE_CLASSIFIER_HEAD:
    print("USE_CLASSIFIER_HEAD is False -> skipping deep-local classifier baseline.")
else:
    def train_local_classifier(silo, seed):
        torch.manual_seed(seed)
        Xtr, ytr = tdata[silo]["train"]; Xva, yva = tdata[silo]["val"]
        head = ProjectionHead(in_dims[silo])
        opt = torch.optim.AdamW(head.parameters(), lr=LR_HEAD, weight_decay=1e-4)
        cw = class_weights_from_labels(ytr) if USE_CLASS_WEIGHTING else None
        best_f1, best_state, no_imp, n = -1.0, None, 0, Xtr.shape[0]
        for r in range(N_ROUNDS):
            head.train(); perm = torch.randperm(n)
            for st in range(0, n, BATCH_SIZE):
                idx = perm[st:st+BATCH_SIZE]; xb = Xtr[idx]; yb = ytr[idx]
                if AUGMENT_NOISE_STD > 0: xb = xb + AUGMENT_NOISE_STD * torch.randn_like(xb)
                opt.zero_grad()
                ce = _mixup_ce(head, xb, yb, class_weights=cw)
                ce.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP); opt.step()
            head.eval()
            with torch.no_grad():
                pred = head.classify(head(Xva)).argmax(-1)
            f1 = f1_score(yva.numpy(), pred.numpy(), average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1; best_state = {k: v.clone() for k, v in head.state_dict().items()}; no_imp = 0
            else:
                no_imp += 1
            if no_imp >= EARLY_STOP_PATIENCE: break
        head.load_state_dict(best_state); return head

    for seed in range(N_SEEDS):
        for silo in tdata:
            h = train_local_classifier(silo, seed); h.eval()
            thr = best_threshold_on_val(h, tdata[silo]["val"][0], tdata[silo]["val"][1])
            with torch.no_grad():
                emb = h(tdata[silo]["test"][0])
                pred = predict_labels(h, emb, None, pos_threshold=thr)
            local_clf_test_f1[silo].append(
                f1_score(tdata[silo]["test"][1].numpy(), pred.cpu().numpy(), average="macro", zero_division=0))
    for silo in tdata:
        m, s = np.mean(local_clf_test_f1[silo]), np.std(local_clf_test_f1[silo])
        print(f"  [local classifier (no FL)] {silo:6s} test macro-F1: {m:.3f} +/- {s:.3f}")


## Stage 1c — Final ablation table: does cross-silo transfer actually lift the weak modality?

In [ ]:
rows = []
for silo in tdata:
    rows.append({
        "silo": silo,
        "standalone": f"{standalone_results[silo][0]:.3f} +/- {standalone_results[silo][1]:.3f}",
        "local_classifier": (f"{np.mean(local_clf_test_f1[silo]):.3f} +/- {np.std(local_clf_test_f1[silo]):.3f}"
                              if local_clf_test_f1[silo] else "n/a"),
        "vanilla_FedProto": f"{np.mean(vanilla_test_f1[silo]):.3f} +/- {np.std(vanilla_test_f1[silo]):.3f}",
        "FedTGP_style": f"{np.mean(tgp_test_f1[silo]):.3f} +/- {np.std(tgp_test_f1[silo]):.3f}",
        "FedProto_beats_standalone": np.mean(vanilla_test_f1[silo]) > standalone_results[silo][0],
        "FedTGP_beats_FedProto": np.mean(tgp_test_f1[silo]) > np.mean(vanilla_test_f1[silo]),
        "FedTGP_beats_local": (np.mean(tgp_test_f1[silo]) > np.mean(local_clf_test_f1[silo])
                                if local_clf_test_f1[silo] else None),
    })
ablation = pd.DataFrame(rows).set_index("silo")
print("="*90, "\nFINAL ABLATION -- macro-F1 (mean +/- std over", N_SEEDS, "seeds)\n" + "="*90)
display(ablation)

print("\nInterpretation:")
print(" - If FedProto_beats_standalone is True for the weak modality (likely EEG/text), that IS")
print("   your thesis's cross-modal transfer claim, evidenced quantitatively.")
print(" - If FedTGP_beats_FedProto is True, the naive-averaging-is-suboptimal-across-modalities")
print("   argument from the Stage-0 report is empirically confirmed in YOUR data, not just cited.")
print(" - If neither FL variant beats standalone for any silo, the projection heads are not yet")
print("   producing an alignable shared space -- revisit head capacity/LR before concluding FL fails.")

ablation.to_csv(f"{OUT}/fl_ablation_results.csv")
summary = {
  "config": {"N_CLIENTS": N_CLIENTS, "LOCAL_EPOCHS": LOCAL_EPOCHS, "N_ROUNDS": N_ROUNDS,
             "EARLY_STOP_PATIENCE": EARLY_STOP_PATIENCE, "N_SEEDS": N_SEEDS,
             "LR_HEAD_swept": LR_HEAD, "HIDDEN_DIM": HIDDEN_DIM, "N_HEAD_LAYERS": N_HEAD_LAYERS,
             "GRAD_CLIP": GRAD_CLIP, "USE_SAMPLE_SIZE_WEIGHTING": USE_SAMPLE_SIZE_WEIGHTING, "USE_CLIENT_RELIABILITY_WEIGHTING": USE_CLIENT_RELIABILITY_WEIGHTING, "WARMUP_ROUNDS": WARMUP_ROUNDS, "USE_ETF_INIT": USE_ETF_INIT},
  "standalone": {s: standalone_results[s] for s in standalone_results},
  "vanilla_rounds_locked": vanilla_rounds_locked, "tgp_rounds_locked": tgp_rounds_locked,
  "ablation": ablation.reset_index().to_dict(orient="records"),
}
json.dump(summary, open(f"{OUT}/fl_fusion_summary.json", "w"), indent=2, default=str)
print("\nsaved:", f"{OUT}/fl_ablation_results.csv", "and", f"{OUT}/fl_fusion_summary.json")

In [ ]:
#    dominated by whichever silo has the most test subjects (here, audio)
def overall_performance(per_silo_means, per_silo_test_sizes):
    unweighted = float(np.mean(list(per_silo_means.values())))
    total_n = sum(per_silo_test_sizes.values())
    weighted = sum(per_silo_means[s] * per_silo_test_sizes[s] for s in per_silo_means) / total_n
    return unweighted, weighted

test_sizes = {silo: len(tdata[silo]["test"][1]) for silo in tdata}
print(f"Test set sizes: {test_sizes}\n")

overall_rows = []
for name, means in [
    ("standalone",      {s: standalone_results[s][0] for s in tdata}),
    ("local_classifier", {s: float(np.mean(local_clf_test_f1[s])) for s in tdata}) if all(local_clf_test_f1[s] for s in tdata) else ("local_classifier", {s: 0.0 for s in tdata}),
    ("vanilla_FedProto", {s: float(np.mean(vanilla_test_f1[s])) for s in tdata}),
    ("FedTGP_style",     {s: float(np.mean(tgp_test_f1[s])) for s in tdata}),
]:
    u, w = overall_performance(means, test_sizes)
    overall_rows.append({"method": name, "overall_unweighted": u, "overall_sample_weighted": w})

overall_df = pd.DataFrame(overall_rows).set_index("method")
print("="*70, "\nOVERALL PERFORMANCE (across all 3 silos)\n" + "="*70)
display(overall_df)
print("\nUnweighted is the more meaningful number for the cross-silo-transfer claim --")
print("it doesn't let audio's larger test set dominate the headline number.")
overall_df.to_csv(f"{OUT}/fl_overall_performance.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
silos_order = list(tdata.keys())
x = np.arange(len(silos_order)); width = 0.25

means_standalone = [standalone_results[s][0] for s in silos_order]
stds_standalone  = [standalone_results[s][1] for s in silos_order]
means_vanilla = [float(np.mean(vanilla_test_f1[s])) for s in silos_order]
stds_vanilla  = [float(np.std(vanilla_test_f1[s])) for s in silos_order]
means_tgp = [float(np.mean(tgp_test_f1[s])) for s in silos_order]
stds_tgp  = [float(np.std(tgp_test_f1[s])) for s in silos_order]

ax.bar(x - width, means_standalone, width, yerr=stds_standalone, label="Standalone", capsize=4)
ax.bar(x,          means_vanilla,   width, yerr=stds_vanilla,   label="Vanilla FedProto", capsize=4)
ax.bar(x + width,  means_tgp,       width, yerr=stds_tgp,       label="FedTGP-style", capsize=4)
ax.axhline(standalone_results[max(standalone_results, key=lambda s: standalone_results[s][0])][0],
           color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Floor to beat (best standalone)")
ax.set_xticks(x); ax.set_xticklabels([s.upper() for s in silos_order])
ax.set_ylabel("Test macro-F1"); ax.set_title("Per-silo performance: standalone vs FL variants\n(mean \u00b1 std over "
              f"{N_SEEDS} seeds)")
ax.legend(loc="upper right", fontsize=9); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_per_silo_performance.png", dpi=120)
plt.show()

fig2, ax2 = plt.subplots(figsize=(9, 5.5))
if vanilla_hist_seed0 is not None:
    ax2.plot(vanilla_hist_seed0["round"], vanilla_hist_seed0["val_macro_f1"],
             label="Vanilla FedProto (seed 0)", marker="o", markersize=3)
if tgp_hist_seed0 is not None:
    ax2.plot(tgp_hist_seed0["round"], tgp_hist_seed0["val_macro_f1"],
             label="FedTGP-style (seed 0)", marker="s", markersize=3)
ax2.set_xlabel("Communication round"); ax2.set_ylabel("Pooled validation macro-F1")
ax2.set_title("Training curves (seed 0): early-stopping trajectory")
ax2.legend(); ax2.grid(alpha=0.3)
fig2.tight_layout()
fig2.savefig(f"{OUT}/fig_training_curves.png", dpi=120)
plt.show()

print(f"Saved: {OUT}/fig_per_silo_performance.png and {OUT}/fig_training_curves.png")
print("NOTE: training curves are for seed 0 only (illustrative) -- the ablation table's")
print("mean+/-std over all 5 seeds remains the number to report, not this single curve.")

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
import numpy as np, torch

assert "tdata" in globals() and "run_fl_training" in globals(), \
    "Run the FL notebook cells above first (same kernel)."

if TASK != "severity3":
    raise RuntimeError(
        f"TASK is '{TASK}'. Set TASK='severity3' in the config cell and re-run the notebook "
        "top-to-bottom before running the per-class cells."
    )

CLASS_NAMES = ["none-minimal", "mild-moderate", "severe"]
assert N_CLASSES == 3, f"Expected 3 classes, got N_CLASSES={N_CLASSES}"
print("OK: 3-class severity task. Silos:", list(tdata.keys()))

## Regenerate heads and collect per-seed predictions

In [ ]:
def collect_predictions(use_trainable_global, n_seeds=N_SEEDS):
    """Returns {silo: {"y_true": arr, "y_pred_per_seed": [arr,...]}} for the given config."""
    out = {s: {"y_true": tdata[s]["test"][1].numpy(), "y_pred_per_seed": []} for s in tdata}
    for seed in range(n_seeds):
        heads, protos, best_round, hist = run_fl_training(
            tdata, in_dims, seed, use_trainable_global=use_trainable_global, lr=LR_HEAD)
        for silo in tdata:
            heads[silo].eval()
            # NOTE: 3-class task -> no pos_threshold (that is binary-only)
            with torch.no_grad():
                emb = heads[silo](tdata[silo]["test"][0])
                pred = predict_labels(heads[silo], emb, protos, pos_threshold=None)
            out[silo]["y_pred_per_seed"].append(
                pred.cpu().numpy() if hasattr(pred, "cpu") else np.asarray(pred))
        print(f"  ({'FedTGP' if use_trainable_global else 'Vanilla'}) seed {seed} done "
              f"(locked round {best_round})")
    return out

print("Collecting Vanilla FedProto predictions...")
vanilla_pred = collect_predictions(use_trainable_global=False)
print("Collecting FedTGP-style predictions...")
fedtgp_pred  = collect_predictions(use_trainable_global=True)
print("done.")

## Per-class F1 table (seed-averaged)

In [ ]:
import pandas as pd

def per_class_table(pred_dict, config_name):
    rows = []
    for silo in tdata:
        yt = pred_dict[silo]["y_true"]
        pcs, macros = [], []
        for yp in pred_dict[silo]["y_pred_per_seed"]:
            pcs.append(f1_score(yt, yp, average=None, labels=[0,1,2], zero_division=0))
            macros.append(f1_score(yt, yp, average="macro", zero_division=0))
        pcs = np.array(pcs)  # (n_seeds, 3)
        rows.append({
            "Silo": silo.upper(),
            "None-minimal F1": round(pcs[:,0].mean(), 3),
            "Mild-moderate F1": round(pcs[:,1].mean(), 3),
            "Severe F1": round(pcs[:,2].mean(), 3),
            "Macro-F1": round(np.mean(macros), 3),
        })
    df = pd.DataFrame(rows)
    print(f"\n=== Per-class F1 ({config_name}) ===")
    print(df.to_string(index=False))
    df.to_csv(f"perclass_f1_3class_{config_name}.csv", index=False)
    return df

vanilla_pc = per_class_table(vanilla_pred, "vanilla")
fedtgp_pc  = per_class_table(fedtgp_pred,  "fedtgp")

# LaTeX for the FedTGP table (swap to vanilla_pc if preferred)
print("\n----- LaTeX (FedTGP-style) -----\n")
print(fedtgp_pc.to_latex(index=False, escape=False, column_format="lcccc",
      caption="Per-class F1 on the 3-class severity task (FedTGP-style, seed-mean).",
      label="tab:perclass_3class_fedtgp"))

## Confusion matrices (pooled across seeds)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def plot_confusions(pred_dict, config_name):
    silos = list(tdata.keys())
    fig, axes = plt.subplots(1, len(silos), figsize=(4.6*len(silos), 4.2))
    if len(silos) == 1: axes = [axes]
    for ax, silo in zip(axes, silos):
        yt = pred_dict[silo]["y_true"]
        # pool: stack true repeated per seed against concatenated preds
        y_true_all = np.concatenate([yt]*len(pred_dict[silo]["y_pred_per_seed"]))
        y_pred_all = np.concatenate(pred_dict[silo]["y_pred_per_seed"])
        cm = confusion_matrix(y_true_all, y_pred_all, labels=[0,1,2])
        ConfusionMatrixDisplay(cm, display_labels=["none-min","mild-mod","severe"]).plot(
            ax=ax, colorbar=False, cmap="Blues", values_format="d")
        ax.set_title(f"{silo.upper()} ({config_name})")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for lbl in ax.get_xticklabels():
            lbl.set_rotation(30); lbl.set_ha("right")
    fig.suptitle(f"3-class severity confusion matrices ({config_name}, pooled over seeds)", y=1.03)
    fig.tight_layout()
    fig.savefig(f"confusion_3class_{config_name}.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"saved: confusion_3class_{config_name}.png")

plot_confusions(fedtgp_pred, "fedtgp")
plot_confusions(vanilla_pred, "vanilla")

## Optional: one-paragraph text summary

In [ ]:
def summarize(pred_dict, config_name):
    for silo in tdata:
        yt = pred_dict[silo]["y_true"]
        pcs = np.mean([f1_score(yt, yp, average=None, labels=[0,1,2], zero_division=0)
                       for yp in pred_dict[silo]["y_pred_per_seed"]], axis=0)
        support = np.bincount(yt, minlength=3)
        best, worst = int(np.argmax(pcs)), int(np.argmin(pcs))
        print(f"{silo.upper()} ({config_name}): strongest on '{CLASS_NAMES[best]}' "
              f"(F1={pcs[best]:.2f}), weakest on '{CLASS_NAMES[worst]}' (F1={pcs[worst]:.2f}); "
              f"test support per class {support.tolist()}.")

summarize(fedtgp_pred, "fedtgp")
summarize(vanilla_pred, "vanilla")

In [ ]:
import pandas as pd, numpy as np

def perclass_table(pc_dict, name):
    rows = []
    for silo in tdata:
        arr = np.array(pc_dict[silo])          # shape (n_seeds, N_CLASSES)
        rows.append({
            "Silo": silo.upper(),
            "None-minimal F1": round(arr[:, 0].mean(), 3),
            "Mild-moderate F1": round(arr[:, 1].mean(), 3),
            "Severe F1": round(arr[:, 2].mean(), 3),
        })
    df = pd.DataFrame(rows)
    print(f"\n=== {name} ===")
    print(df.to_string(index=False))
    return df

perclass_table(vanilla_perclass, "Vanilla FedProto (per-class F1)")
perclass_table(tgp_perclass, "FedTGP-style (per-class F1)")